# Simulation: the firmware on the laptop
The ESP32 isn't connected yet, so the firmware runs in the MicroPython Unix port first. The code in `firmware/` is exactly the code that will be copied to the board. Only `emlearn_fft` is replaced by a copy of its FFT in Python (`sim/run_sim.py`), because emlearn has no build for this Mac.

## Day 8 – the 12 features in MicroPython
`firmware/features.py` computes the same 12 features as `src/features.py`, but sample by sample from a ring buffer, with pre-allocated `array` buffers and no numpy. Today I check that both give the same numbers.

### Setup
`run_firmware(x, op)` writes the signal to a file, starts MicroPython and feeds the samples one by one into a ring buffer of 1024 samples per axis, the way they arrive on the board. Every 256 samples it computes one window, so it returns one row of 12 features per window, like `features(x, op)`.

In [1]:
import sys
sys.path.insert(0, '..')
import time
import numpy as np, pandas as pd
from sim.run_sim import run_alarm, run_firmware, run_scores
from src.features import FREQ, NAMES, SPEED, WIN, features, index_recordings, load

idx = index_recordings()

### The spindle band as FFT bins
On the laptop the band is every FFT bin within ±10% of the spindle speed, found with a float comparison. The board uses integer maths instead: the first bin at or above 0.9 × speed and the last bin at or below 1.1 × speed. This cell prints the laptop bins, which should be 93–112 for 200 Hz and 116–140 for 250 Hz, the values the integer formula in `spindle_bins` gives.

In [2]:
for s in sorted(set(SPEED.values())):
    near = np.flatnonzero(np.abs(FREQ - s) <= .1 * s)
    lo, hi = (9 * s * WIN + 10 * 2000 - 1) // (10 * 2000), 11 * s * WIN // (10 * 2000)
    print(f'{s} Hz: laptop bins {near[0]}-{near[-1]}, firmware formula {lo}-{hi}')

200 Hz: laptop bins 93-112, firmware formula 93-112
250 Hz: laptop bins 116-140, firmware formula 116-140


### Test signals
- **Sine:** 250 Hz with amplitude 100 counts and an offset of -1000 counts, where the answer is known: RMS 70.7, peak-to-peak 200, crest factor 1.414.
- **Flat:** all zeros. RMS is 0, so the crest factor divides by EPS only, and the spindle energy is log10 of EPS only.
- **Real recordings:** complete good and bad recordings of OP01 (250 Hz) and OP05 (200 Hz), so both band widths are tested.

In [3]:
t = np.arange(2000 * 5) / 2000
sine = np.tile(100 * np.sin(2 * np.pi * 250 * t)[:, None] - 1000, (1, 3))
cases = {'sine 250 Hz': (sine, 'OP01'), 'flat': (np.zeros((WIN + 4 * 256, 3)), 'OP01')}
for op in ['OP01', 'OP05']:
    for lab in ['good', 'bad']:
        r = idx[(idx['op'] == op) & (idx['label'] == lab)].iloc[0]
        cases[f"{op} {lab} {r['period']} {r['num']}"] = (load(r['path']), op)
for name, (x, op) in cases.items():
    print(f'{name:24} {len(x) / 2000:5.1f} s')

sine 250 Hz                5.0 s
flat                       1.0 s
OP01 good 2019-08 000     30.2 s
OP01 bad 2019-08 000      19.5 s
OP05 good 2019-08 000     23.0 s
OP05 bad 2019-08 000      13.4 s


### Firmware against laptop
For every test signal: features from MicroPython and from numpy, then the largest absolute difference per feature, over all windows. RMS and peak-to-peak are in counts, crest factor has no unit and the spindle energy is in dB.

In [4]:
gaps, n = [], 0
start = time.time()
for name, (x, op) in cases.items():
    fw, lap = run_firmware(x, op), features(x, op)
    assert fw.shape == lap.shape, name
    gaps.append(pd.Series(np.abs(fw - lap).max(axis=0), NAMES, name=name))
    n += len(fw)
gaps = pd.DataFrame(gaps)
print(f'{n} windows in {time.time() - start:.0f} s')
print(gaps.T.to_string(float_format='{:.1e}'.format))

701 windows in 11 s
              sine 250 Hz    flat  OP01 good 2019-08 000  OP01 bad 2019-08 000  OP05 good 2019-08 000  OP05 bad 2019-08 000
rms X             7.6e-06 0.0e+00                3.0e-05               3.0e-05                2.9e-05               1.5e-05
rms Y             7.6e-06 0.0e+00                1.4e-05               1.3e-05                1.2e-05               7.5e-06
rms Z             7.6e-06 0.0e+00                1.5e-05               1.3e-05                7.6e-06               7.5e-06
p2p X             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
p2p Y             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
p2p Z             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
crest X           1.2e-07 0.0e+00                4.2e-07               2.3e-07                8.3e-07           

### Sine check
The firmware values of the first sine window against the expected values.

In [5]:
fw = run_firmware(sine, 'OP01')
print(pd.Series(fw[0], NAMES).round(4).to_string())

rms X            70.710701
rms Y            70.710701
rms Z            70.710701
p2p X           200.000000
p2p Y           200.000000
p2p Z           200.000000
crest X           1.414200
crest Y           1.414200
crest Z           1.414200
spindle_db X     89.921501
spindle_db Y     89.921501
spindle_db Z     89.921501


### What Day 8 shows
- **The firmware gives the laptop's numbers.** Over 701 windows of six signals, the largest difference is 3.0e-05 counts for RMS, 0 for peak-to-peak, 8.3e-07 for the crest factor and 6.7e-06 dB for the spindle energy. These differences come from storing the results as float32. The model's threshold sits several dB above normal, so differences this small can't change an alarm.
- **The spindle band is the same.** The integer formula on the board picks bins 93–112 for 200 Hz and 116–140 for 250 Hz, the same bins as the laptop.
- **The sine gives the expected values:** RMS 70.71, peak-to-peak 200 and crest factor 1.414, with the -1000 offset removed.
- **Limits of this check:** the Mac's MicroPython calculates in double precision, while the ESP32-S3 uses single precision, so the board may differ a little more. The FFT here is the Python copy of emlearn's FFT, not emlearn itself. Both have to be checked again on the board, and the time on the laptop says nothing about the time on the chip.

## Day 9 – the Mahalanobis score in MicroPython
Day 8 gave the 12 features on the board. Today the board also gets the model: the Mahalanobis distance from the period centre, divided by the operation's threshold, so 1 is the threshold again. `tools/stream.py` will send the operation and the period, so the board loads the matching centre. The 5 s median, the idle filter and the 3 s LED hold come later with `firmware/main.py`.

### Setup
The model functions from notebook 03 (Days 5 and 6) now live in `src/models.py`: the split by operation and period, the fit with a centre per period, the score and the threshold. This cell computes the features of all 447 recordings again (about 20 s) and makes the same split as notebook 03.

In [6]:
from src.models import fit, score, split_by_period, threshold

F = {i: features(load(p), op) for i, (p, op) in enumerate(zip(idx['path'], idx['op']))}
idx['split'] = split_by_period(idx)
print(idx.groupby(['split', 'label']).size().unstack(fill_value=0))

label  bad  good
split           
test    34    81
train    0   244
val      0    88


### Refit and check against notebook 03
One model per operation, fitted on the train recordings, threshold from the validation recordings. If the code moved to `src/` correctly, the thresholds are the ones printed on Day 6 of notebook 03.

In [7]:
DAY6 = {'OP01': 3.95, 'OP02': 3.48, 'OP03': 4.15, 'OP04': 4.52, 'OP05': 3.26, 'OP06': 5.14, 'OP07': 3.54,
        'OP08': 3.9, 'OP09': 4.89, 'OP10': 3.58, 'OP11': 4.88, 'OP12': 3.89, 'OP14': 3.7}  # notebook 03, Day 6
M = {}
for op, g in idx.groupby('op'):
    M[op] = fit(F, g[g['split'] == 'train'])
    M[op]['thr'] = threshold(M[op], F, g[g['split'] == 'val'])
thr = pd.Series({op: m['thr'] for op, m in M.items()})
print(pd.DataFrame({'now': thr.round(2), 'day 6': pd.Series(DAY6)}).T.to_string())
assert (thr.round(2) == pd.Series(DAY6)).all()

       OP01  OP02  OP03  OP04  OP05  OP06  OP07  OP08  OP09  OP10  OP11  OP12  OP14
now    3.95  3.48  4.15  4.52  3.26  5.14  3.54   3.9  4.89  3.58  4.88  3.89   3.7
day 6  3.95  3.48  4.15  4.52  3.26  5.14  3.54   3.9  4.89  3.58  4.88  3.89   3.7


### Export firmware/model.bin
One record for every operation and period that has a centre. Each record is a 12-byte key such as `OP01 2019-08`, then 169 float32 numbers: the centre (12), the std (12), the inverse covariance (12 × 12) and the threshold (1). Every record has the same size, so the board can read them one after another into one buffer and stop at the key it was sent. float32 is the number format of the board, so rounding to it happens here, once.

In [8]:
rec = []
for op, m in M.items():
    for period, c in m['centre'].items():
        key = f'{op} {period}'.encode()
        assert len(key) == 12
        rec.append(key + np.r_[c, m['std'], m['inv'].ravel(), m['thr']].astype('<f4').tobytes())
with open('../firmware/model.bin', 'wb') as f:
    f.write(b''.join(rec))
print(len(rec), 'records of', len(rec[0]), 'bytes =', len(rec) * len(rec[0]), 'bytes')

59 records of 688 bytes = 40592 bytes


### Firmware score against laptop score
For every operation, one good and one bad test recording, both from a period with a centre. MicroPython computes features and score sample by sample (`run_scores`), numpy computes `score / threshold` from the laptop features. The firmware scores every window, so idle windows are compared too. Besides the largest difference, the cell counts windows where the two disagree about being above the threshold (score > 1).

In [9]:
test = idx[idx['split'] == 'test']
rows, start = [], time.time()
for op, g in test.groupby('op'):
    for lab in ['good', 'bad']:
        i = next(i for i in g.index[g['label'] == lab] if g.loc[i, 'period'] in M[op]['centre'])
        x, period = load(idx.loc[i, 'path']), idx.loc[i, 'period']
        fw = run_scores(x, op, period)
        lap = score(M[op], features(x, op), period) / M[op]['thr']
        rows.append(dict(op=op, label=lab, period=period, windows=len(fw), max_score=lap.max(),
                         max_diff=np.abs(fw - lap).max(), max_rel_diff=(np.abs(fw - lap) / lap).max(),
                         disagree=((fw > 1) != (lap > 1)).sum()))
cmp = pd.DataFrame(rows)
print(f'{cmp.windows.sum()} windows in {time.time() - start:.0f} s')
print(cmp.to_string(index=False, float_format='{:.2e}'.format))

9769 windows in 155 s
  op label  period  windows  max_score  max_diff  max_rel_diff  disagree
OP01  good 2019-08      229   3.34e+00  5.33e-07      1.39e-06         0
OP01   bad 2019-08      149   1.62e+00  3.43e-07      3.53e-07         0
OP02  good 2019-08      341   2.57e+00  7.75e-07      8.70e-07         0
OP02   bad 2019-02      277   5.09e+00  1.08e-06      3.79e-07         0
OP03  good 2019-08      693   4.78e+00  7.15e-07      1.21e-06         0
OP03   bad 2019-08      542   1.90e+00  5.16e-07      6.56e-07         0
OP04  good 2019-08      533   3.06e+00  6.07e-07      1.23e-06         0
OP04   bad 2019-08      261   2.77e+00  8.66e-07      5.87e-07         0
OP05  good 2019-08      177   2.57e+00  5.44e-07      6.54e-07         0
OP05   bad 2019-08      101   2.45e+00  5.10e-07      4.53e-07         0
OP06  good 2019-08      621   3.84e+00  3.86e-07      9.70e-07         0
OP06   bad 2019-08      661   1.40e+00  5.35e-07      9.07e-07         0
OP07  good 2019-08      193  

### What Day 9 shows
- **The move to `src/models.py` changed nothing.** The refit gives the same split and the same 13 thresholds as Day 6 of notebook 03, from 3.26 (OP05) to 5.14 (OP06).
- **The model is small.** `model.bin` has 59 records (one per operation and period with a centre) of 688 bytes, 40,592 bytes in total. The board only ever holds one record in RAM: 169 float32 numbers, 676 bytes.
- **The firmware gives the laptop's score.** Over 9,769 windows of 26 test recordings, the largest difference is 1.1e-06 on the score / threshold scale, at most 2.0e-06 relative to the score. No window is above the threshold on one side and below it on the other.
- **These are single-window scores.** Good recordings go above 1 here too, because the threshold was set on the 5 s median score, not on single windows. The median, the idle filter and the 3 s LED hold come with `firmware/main.py`.
- **Limits of this check:** the same as Day 8. The Mac's MicroPython calculates in double precision and the ESP32-S3 in single precision, and OP10 2021-08 has no record, so the board can't score it.

## Day 10 – the LED rule in MicroPython
Day 9 put the window score on the board. What the person at the machine sees is the LED, and notebook 03 switches it on in three steps after the score: drop idle windows, take the median of the last 40 scores (5 s), and turn the LED on once that median has stayed above 1 for 24 windows (3.1 s). Today these three steps go into `firmware/model.py`, and I check that the LED goes on in exactly the same windows as on the laptop.

### The LED rule on the board
The laptop has the whole recording as an array. The board gets one window at a time, so `update(x)` keeps a little state between windows:
- **Idle:** if the largest axis RMS is below 30 counts, the window is skipped. It doesn't enter the median and the LED stays as it was, the same as dropping idle windows on the laptop.
- **Median:** the last 40 scores sit in a ring of 40 floats. For the median they're copied into a second array and sorted with insertion sort. With 40 values that's fast enough, and it creates no new arrays. 40 is even, so the median is the mean of the two middle values, like `np.median`.
- **Hold:** a counter of windows in a row with the median above 1. It goes back to 0 at the first window below 1, and the LED is on while it's at least 24.

`load()` resets the state, so every new operation starts with an empty median, the same as each recording on the laptop. `held` moved from notebook 03 to `src/models.py`.

### Firmware LED against laptop LED on the whole test set
Every test recording from a period with a centre (114, the same as Day 7). MicroPython runs features, score, median and LED sample by sample (`run_alarm`). The laptop computes the Day 7 LED from the features without idle windows. The cell compares the two on the non-idle windows: the largest difference of the 5 s median, and the number of windows where one LED is on and the other off. This takes about 13 minutes.

In [10]:
from src.features import STEP, is_idle
from src.models import held, median_score, running

test10 = idx[(idx['split'] == 'test') & np.array([p in M[o]['centre'] for o, p in zip(idx['op'], idx['period'])])]
fw_led, rows, start = {}, [], time.time()
for i in test10.index:
    op, period = idx.loc[i, 'op'], idx.loc[i, 'period']
    med_fw, led_fw = run_alarm(load(idx.loc[i, 'path']), op, period)
    keep = ~is_idle(F[i])
    med_fw, led_fw = med_fw[keep], led_fw[keep]
    med_lap = median_score(score(M[op], running(F[i]), period) / M[op]['thr'])
    fw_led[i] = led_fw
    rows.append(dict(med_diff=np.nanmax(np.abs(med_fw - med_lap)), same_nan=(np.isnan(med_fw) == np.isnan(med_lap)).all(),
                     led_differ=(led_fw != held(med_lap)).sum()))
cmp10 = pd.DataFrame(rows, test10.index)
print(f'{len(test10)} recordings, {sum(len(v) for v in fw_led.values())} non-idle windows in {(time.time() - start) / 60:.1f} min')
print(f"largest median difference {cmp10['med_diff'].max():.1e}, median not ready in the same windows: {cmp10['same_nan'].all()}, "
      f"windows with a different LED: {cmp10['led_differ'].sum()}")

114 recordings, 43640 non-idle windows in 11.5 min
largest median difference 1.3e-06, median not ready in the same windows: True, windows with a different LED: 0


### Day 7 results from the firmware LED
The same measures as Day 7 in notebook 03, now from the LED the firmware switched: bad recordings caught, good recordings that switch the LED on, LED starts per hour of normal signal, and the median delay from the start of a bad file to the first LED-on. Day 7 had 28 of 33, 3 of 81, 6.9 per hour and 8.4 s.

In [11]:
bad, good = test10.index[test10['label'] == 'bad'], test10.index[test10['label'] == 'good']
starts = sum(np.sum(np.diff(fw_led[i].astype(int), prepend=0) == 1) for i in good)
hours = sum(len(fw_led[i]) for i in good) * STEP / 2000 / 3600
delay = [(np.argmax(fw_led[i]) * STEP + WIN) / 2000 for i in bad if fw_led[i].any()]
print(f"firmware LED: caught {sum(fw_led[i].any() for i in bad)} of {len(bad)}, false flags {sum(fw_led[i].any() for i in good)} of {len(good)}, "
      f"false alarms/h {starts / hours:.1f}, median delay {np.median(delay):.1f} s")

firmware LED: caught 28 of 33, false flags 3 of 81, false alarms/h 6.9, median delay 8.4 s


### Log to metrics.csv
One recording-level row for the firmware LED, in the same form as the Day 7 row of notebook 03. The recording AUROC uses each recording's share of windows with the LED on, as on Day 7. Quantization is float32, because the model and the features are stored as float32 on the board.

In [12]:
import csv, datetime
from sklearn.metrics import roc_auc_score

rec_auroc = roc_auc_score(test10['label'] == 'bad', [fw_led[i].mean() for i in test10.index])
with open('../results/metrics.csv', 'a', newline='') as f:
    csv.writer(f).writerow([datetime.date.today(), '04_simulation.ipynb', 'day10 firmware LED in MicroPython', 'recording by period',
                            'recording', 'maha', 12, 'float32', '', round(rec_auroc, 4), '', '',
                            round(starts / hours, 1), round(np.median(delay), 2), ''])
print(pd.read_csv('../results/metrics.csv')[['source', 'step', 'quantization', 'AUROC', 'false_alarms_per_hour', 'detection_delay_s']].tail(2).to_string())

                 source                               step quantization  AUROC  false_alarms_per_hour  detection_delay_s
12      03_models.ipynb    day7 LED after 24 windows above        float  0.908                    6.9               8.45
13  04_simulation.ipynb  day10 firmware LED in MicroPython      float32  0.908                    6.9               8.45


### What Day 10 shows
- **The board switches the LED exactly like the laptop.** Over all 114 calibrated test recordings (43,640 non-idle windows), the firmware LED and the Day 7 LED differ in 0 windows. The 5 s median differs by at most 1.3e-06, and it becomes ready in the same window on both.
- **So the Day 7 results hold for the firmware:** 28 of 33 bad recordings caught, 3 of 81 good ones switch the LED on, 6.9 LED starts per hour of normal signal and a median delay of 8.4 s.
- **The rule needs little memory.** Two arrays of 40 floats and two counters. The median sorts a copy with insertion sort, so no array is created per window. MicroPython still makes small float objects during the arithmetic, which is why `main.py` will call `gc.collect()` between windows.
- **Limits of this check:** the same as Days 8 and 9. This is the Unix port in double precision with a Python stand-in for emlearn's FFT, not the ESP32-S3. The 11.5 min on the Mac also says nothing about the time per window on the chip. That is measured on the board in Week 4.

## Day 11 – the main loop: firmware/main.py
Days 8 to 10 put features, score and LED rule into `firmware/`, but something still has to feed them samples and show the result. That's `firmware/main.py`, the file the board runs after it starts. Today I write it and check that the full loop, from the received samples to the serial line, gives the same score and LED as the laptop.

### What main.py does
- **Start:** join the WiFi with the login from `firmware/secrets.py` (gitignored), switch the LED off, open a TCP server on port 5005 and print `gc.mem_free()`. All buffers are allocated here, before the loop: the 13-byte header, 256 samples of int16, three rings of 1024 floats and the 12 features.
- **One recording = one connection.** `tools/stream.py` will first send a header like `OP01 2019-08\n`, so the board loads the right record from `model.bin`, and then X, Y, Z per sample as little-endian int16. The data are whole numbers between −6668 and 6074 counts, so int16 (±32767) loses nothing, and 2000 samples/s are 12 kB/s.
- **Every 256 samples:** `readinto` waits until the 256 new samples are in. They're copied into the rings, and from 1024 samples on the board computes the window features, `model.update` and the LED. Then it prints `score,alarm,latency_us` and calls `gc.collect()`. The time runs from the moment the samples are in until the line is ready, so it covers the work on the chip and not the waiting for WiFi.
- **LED:** the onboard LED of the DevKitC-1 is an RGB LED (WS2812), so it's driven with `neopixel`. Its pin is GPIO48 on board v1.0 and GPIO38 on v1.1. That's still UNVERIFIED until I've checked my board.

In the simulation, `sim/run_sim.py` gives `main.py` stand-ins for `network`, `machine`, `neopixel` and `secrets`. WiFi counts as connected at once and the server runs on 127.0.0.1. `run_main` then sends the recordings over a real TCP connection and reads the serial lines.

### Setup
Rebuilds the state from Day 9 (index, split, features and the 13 models), because this check needs the test split and the laptop scores. The thresholds have to come out the same as on Day 9 again.

In [ ]:
from sim.run_sim import run_main
from src.features import is_idle
from src.models import fit, held, median_score, running, score, split_by_period, threshold

F = {i: features(load(p), op) for i, (p, op) in enumerate(zip(idx['path'], idx['op']))}
idx['split'] = split_by_period(idx)
M = {}
for op, g in idx.groupby('op'):
    M[op] = fit(F, g[g['split'] == 'train'])
    M[op]['thr'] = threshold(M[op], F, g[g['split'] == 'val'])
print({op: round(m['thr'], 2) for op, m in M.items()})

{'OP01': np.float64(3.95), 'OP02': np.float64(3.48), 'OP03': np.float64(4.15), 'OP04': np.float64(4.52), 'OP05': np.float64(3.26), 'OP06': np.float64(5.14), 'OP07': np.float64(3.54), 'OP08': np.float64(3.9), 'OP09': np.float64(4.89), 'OP10': np.float64(3.58), 'OP11': np.float64(4.88), 'OP12': np.float64(3.89), 'OP14': np.float64(3.7)}


### main.py against the laptop
The same 26 test recordings as Day 9: one good and one bad per operation, from a period with a centre. All 26 go through one running `main.py`, one connection each. Per recording, the cell compares the score with the numpy score in every window, and the LED with the Day 7 LED in the non-idle windows (an idle window leaves the LED as it was, as on Day 10). `main.py` prints the score with 6 decimals, so differences up to 5e-07 are just that rounding.

In [ ]:
test = idx[idx['split'] == 'test']
pick = [next(i for i in g.index[g['label'] == lab] if g.loc[i, 'period'] in M[op]['centre'])
        for op, g in test.groupby('op') for lab in ['good', 'bad']]
start = time.time()
out = run_main([(load(idx.loc[i, 'path']), idx.loc[i, 'op'], idx.loc[i, 'period']) for i in pick])
rows = []
for i, o in zip(pick, out):
    op, period = idx.loc[i, 'op'], idx.loc[i, 'period']
    lap = score(M[op], F[i], period) / M[op]['thr']
    keep = ~is_idle(F[i])
    led_lap = held(median_score(score(M[op], running(F[i]), period) / M[op]['thr']))
    rows.append(dict(op=op, label=idx.loc[i, 'label'], windows=len(o), same_count=len(o) == len(lap),
                     max_diff=np.abs(o[:, 0] - lap).max(), led_on=int(o[keep, 1].sum()), led_differ=int(((o[keep, 1] > 0) != led_lap).sum())))
cmp11 = pd.DataFrame(rows)
print(f'{cmp11.windows.sum()} windows in {time.time() - start:.0f} s')
print(cmp11.to_string(index=False, float_format='{:.1e}'.format))

9769 windows in 152 s
  op label  windows  same_count  max_diff  led_on  led_differ
OP01  good      229        True   8.8e-07       0           0
OP01   bad      149        True   6.8e-07      87           0
OP02  good      341        True   1.0e-06       0           0
OP02   bad      277        True   1.3e-06     215           0
OP03  good      693        True   1.1e-06       0           0
OP03   bad      542        True   8.3e-07     480           0
OP04  good      533        True   8.7e-07       0           0
OP04   bad      261        True   1.1e-06     142           0
OP05  good      177        True   1.0e-06       0           0
OP05   bad      101        True   8.2e-07      39           0
OP06  good      621        True   6.8e-07       0           0
OP06   bad      661        True   8.7e-07       0           0
OP07  good      193        True   9.8e-07       0           0
OP07   bad      122        True   9.0e-07      60           0
OP08  good      293        True   9.9e-07       

### Samples in small pieces
Over WiFi the samples don't arrive in neat blocks of 256. This sends the first bad recording again, in pieces of 7 bytes, so a sample is split across two pieces most of the time. If `readinto` really waits for the full block, the serial lines are the same as above, apart from the time column.

In [ ]:
i = pick[1]
small = run_main([(load(idx.loc[i, 'path']), idx.loc[i, 'op'], idx.loc[i, 'period'])], piece=7)[0]
print(idx.loc[i, 'op'], idx.loc[i, 'label'], f'{len(small)} windows, score and LED the same as in one piece: {np.array_equal(small[:, :2], out[1][:, :2])}')

OP01 bad 149 windows, score and LED the same as in one piece: True


### What Day 11 shows
- **The whole firmware gives the laptop's result.** `main.py` received 26 test recordings over TCP, one per connection, and printed 9,769 serial lines, one for every window the laptop has. The score differs from numpy by at most 1.3e-06: Day 9 had 1.1e-06, and printing 6 decimals adds up to 5e-07. The LED differs in 0 non-idle windows. It stays off in the bad OP06, OP08 and OP09 recordings, the same misses as on Day 7.
- **Split packets don't matter.** Sent in 7-byte pieces, the bad OP01 recording gives the same scores and LED as in one piece, because `readinto` waits for the full 256 samples.
- **Allocation:** all arrays are created before the loop. Inside it, the floats in the arithmetic and the printed line are still small objects, and `gc.collect()` after each window clears them away.
- **Not checked yet:** WiFi, the LED pin (GPIO48 or GPIO38, UNVERIFIED) and the time per window. The stand-ins skip the first two, and the latency column on the Mac says nothing about the ESP32-S3. All three need the board, and sending at the real 2000 samples/s needs `tools/stream.py`.